# DC-AE 重建评估

加载训练好的 DC-AE，在指定 split 上计算重建指标并可视化。

**指标层次**：
1. 全局指标（MSE / MAE / RMSE / R² / PSNR）
2. **按变量分组指标**（U / V / T / S / SSH 各自的指标）
3. **逐通道 Pearson 相关**（检测符号翻转：r ≈ -1 表示整体反转）

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '../..'))
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

from model.dcae import DCAE
from data.dataset import build_dataset, load_constants
from data.data_utils import normalize_fn
from config import get_dataset_config

DEVICE = torch.device('cpu')  # DCAE 参数量大，CPU 评估避免显存溢出

# ── 可修改配置 ──────────────────────────────────────────────────────────────
DATA_NAME   = 'glorys12_kuroshio_extension'
TAG         = 'dcae_bc64_cm1248_lc16_fft0.5'
SPLIT       = 'val'   # train / val / test
BATCH_SIZE  = 4
NUM_WORKERS = 4
MAX_BATCHES = 10      # None = 全量；与 eval_vqvae 保持一致

# DC-AE 结构（与训练保持一致）
BASE_CHANNELS         = 64
CHANNEL_MULTIPLIERS   = [1, 2, 4, 8]
LATENT_CHANNELS       = 16
NUM_RES_BLOCKS        = 2
ATTENTION_RESOLUTIONS = [1, 2]
NUM_HEADS             = 8
# ────────────────────────────────────────────────────────────────────────────

CKPT_PATH = os.path.join(PROJECT_ROOT, 'output', DATA_NAME, TAG, 'best_model.pth')
assert os.path.exists(CKPT_PATH), f'Checkpoint not found: {CKPT_PATH}'
print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'DEVICE:       {DEVICE}')
print(f'CKPT_PATH:    {CKPT_PATH}')

In [ ]:
# ── 构建数据集与常量 ─────────────────────────────────────────────────────────
dataset_config = get_dataset_config(DATA_NAME)

_date_ranges = {
    'train': dataset_config.train_date_range,
    'val':   dataset_config.val_date_range,
    'test':  dataset_config.test_date_range,
}
dataset = build_dataset(dataset_config.raw_data_dir, _date_ranges[SPLIT])
loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False,   # shuffle=False，与 eval_vqvae 一致
                     num_workers=NUM_WORKERS, pin_memory=False)

# constants = (normed_ocean_mean, normed_ocean_std, raw_ocean_min, raw_ocean_max, depths, mask)
constants = load_constants(dataset_config.constant_dir)
mu    = constants[0][..., None, None].to(DEVICE)  # [C, 1, 1]
sigma = constants[1][..., None, None].to(DEVICE)  # [C, 1, 1]
mask  = constants[-1].float()                      # [C, H, W]  (1=ocean, 0=land)

print(f'Dataset [{SPLIT}]: {len(dataset)} samples')
print(f'Channels: {dataset_config.num_channels},  mask shape: {tuple(mask.shape)}')
print(f'mu shape: {tuple(mu.shape)},  sigma shape: {tuple(sigma.shape)}')

In [ ]:
# ── 构建并加载模型 ───────────────────────────────────────────────────────────
def build_and_load_model(ckpt_path, in_channels, device):
    ckpt  = torch.load(ckpt_path, map_location=device)
    state = ckpt.get('model_state_dict', ckpt.get('model', ckpt))
    # 去除 DDP 的 'module.' 前缀
    state = {(k[7:] if k.startswith('module.') else k): v for k, v in state.items()}

    model = DCAE(
        in_channels=in_channels,
        base_channels=BASE_CHANNELS,
        channel_multipliers=CHANNEL_MULTIPLIERS,
        latent_channels=LATENT_CHANNELS,
        num_res_blocks=NUM_RES_BLOCKS,
        attention_resolutions=ATTENTION_RESOLUTIONS,
        num_heads=NUM_HEADS,
    ).to(device)
    model.load_state_dict(state, strict=True)
    model.eval()
    return model

model = build_and_load_model(CKPT_PATH, dataset_config.num_channels, DEVICE)
print(f'Model loaded.  Parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# ── 评估工具函数 ─────────────────────────────────────────────────────────────

# 变量分组定义
VAR_GROUPS = {
    'U':   list(range(0, 25)),
    'V':   list(range(25, 50)),
    'T':   list(range(50, 75)),
    'S':   list(range(75, 100)),
    'SSH': [100],
}

CHANNEL_LABELS = {}
for vi, var in enumerate(['U', 'V', 'T', 'S']):
    for lev in range(25):
        CHANNEL_LABELS[vi * 25 + lev] = f"{var}_L{lev}"
CHANNEL_LABELS[100] = "SSH"


def masked_metrics(pred, target, mask):
    """MSE / MAE / RMSE / R² / PSNR，只在掩膜有效（ocean）像素上计算。

    pred, target : [B, C, H, W]  float32，无 NaN
    mask         : 可广播到 [B, C, H, W]，0/1
    返回长度为 B 的 list of dict。
    """
    eps = 1e-8
    m = mask.to(pred.device)
    while m.ndim < pred.ndim:
        m = m.unsqueeze(0)
    m = m.expand_as(pred)

    B     = pred.shape[0]
    flat  = lambda t: t.reshape(B, -1)

    n_valid  = flat(m).sum(1).clamp_min(eps)                # [B]
    diff     = (pred - target) * m
    sq_err   = flat(diff.pow(2)).sum(1)                     # [B]
    abs_err  = flat(diff.abs()).sum(1)                      # [B]
    mse      = sq_err / n_valid
    mae      = abs_err / n_valid
    rmse     = mse.sqrt()

    # R²：ss_tot 仅统计 ocean 像素的方差
    mean_t  = flat(target * m).sum(1) / n_valid             # [B]
    ss_tot  = flat((target - mean_t.view(B, 1, 1, 1)).pow(2) * m).sum(1).clamp_min(eps)
    r2      = 1.0 - sq_err / ss_tot

    # PSNR：数据范围取 ocean 像素的 max-min
    INF     = 1e10
    oc_max  = (target * m + (1 - m) * (-INF)).reshape(B, -1).max(1).values
    oc_min  = (target * m + (1 - m) *   INF ).reshape(B, -1).min(1).values
    d_range = (oc_max - oc_min).clamp_min(eps)
    psnr    = 20.0 * torch.log10(d_range / rmse.clamp_min(eps))

    return [
        {'mse':  mse[i].item(),  'mae':  mae[i].item(),
         'rmse': rmse[i].item(), 'r2':   r2[i].item(),
         'psnr': psnr[i].item()}
        for i in range(B)
    ]


def per_channel_correlation(pred, target, mask):
    """逐通道 Pearson 相关系数。

    pred, target : [B, C, H, W]
    mask         : broadcastable to [B, C, H, W]
    返回 [B, C] tensor。
    """
    m = mask.to(pred.dtype)
    while m.ndim < pred.ndim:
        m = m.unsqueeze(0)
    m = m.expand_as(pred)

    B, C, H, W = pred.shape
    p_flat = (pred * m).view(B, C, -1)
    t_flat = (target * m).view(B, C, -1)
    m_flat = m.view(B, C, -1)
    n_valid = m_flat.sum(2).clamp_min(1)

    p_mean = p_flat.sum(2) / n_valid
    t_mean = t_flat.sum(2) / n_valid
    p_c = p_flat - p_mean[:, :, None]
    t_c = t_flat - t_mean[:, :, None]

    cov   = (p_c * t_c * m_flat).sum(2) / n_valid
    std_p = ((p_c.pow(2) * m_flat).sum(2) / n_valid).sqrt()
    std_t = ((t_c.pow(2) * m_flat).sum(2) / n_valid).sqrt()
    corr  = cov / (std_p * std_t).clamp_min(1e-8)
    return corr


@torch.inference_mode()
def evaluate(model, loader, mu, sigma, mask, device, max_batches=None):
    """遍历数据集，返回 (per-sample metrics DataFrame, per-channel corr, vis_cache)。"""
    records   = []
    var_records = []
    all_corrs = []
    vis_cache = None

    for step, batch in enumerate(loader):
        if max_batches is not None and step >= max_batches:
            break

        raw = batch.to(device=device, dtype=torch.float32)
        x_norm = normalize_fn(raw, mu=mu, sigma=sigma)
        recon_norm = model(x_norm)

        x_phys     = (x_norm     * sigma + mu).cpu()
        recon_phys = (recon_norm * sigma + mu).cpu()
        B = x_phys.shape[0]

        # 全局指标
        for rec in masked_metrics(recon_phys, x_phys, mask):
            rec['step'] = step
            records.append(rec)

        # 按变量分组指标
        for var_name, ch_indices in VAR_GROUPS.items():
            for rec in masked_metrics(
                recon_phys[:, ch_indices], x_phys[:, ch_indices],
                mask[ch_indices] if mask.ndim == 3 else mask
            ):
                rec['variable'] = var_name
                rec['step'] = step
                var_records.append(rec)

        # 逐通道相关系数
        corr = per_channel_correlation(recon_phys, x_phys, mask)  # [B, C]
        all_corrs.append(corr)

        if vis_cache is None:
            vis_cache = {'x_phys': x_phys.clone(), 'recon_phys': recon_phys.clone()}

    corr_all = torch.cat(all_corrs, dim=0)  # [N, C]
    return pd.DataFrame(records), pd.DataFrame(var_records), corr_all, vis_cache

In [ ]:
# ── 运行评估 ─────────────────────────────────────────────────────────────────
df, df_var, corr_all, vis_cache = evaluate(model, loader, mu, sigma, mask, DEVICE, max_batches=MAX_BATCHES)

metric_cols = ['mse', 'mae', 'rmse', 'r2', 'psnr']
print(f'Evaluated {len(df)} samples\n')

print('=== 全局指标 ===')
print(df[metric_cols].mean().to_frame('mean').T.round(4).to_string())

print('\n=== 按变量分组指标 ===')
var_summary = df_var.groupby('variable')[metric_cols].mean().round(6)
print(var_summary.to_string())

# 逐通道相关系数（样本平均）
corr_mean = corr_all.mean(dim=0).numpy()  # [C]
print(f'\n=== 逐通道相关系数摘要 ===')
for var_name, ch_indices in VAR_GROUPS.items():
    var_corr = corr_mean[ch_indices]
    neg_chs = [(ch, corr_mean[ch]) for ch in ch_indices if corr_mean[ch] < 0.9]
    status = f'min={var_corr.min():.4f}, mean={var_corr.mean():.4f}'
    if neg_chs:
        status += f'  ⚠ {len(neg_chs)} channels < 0.9'
    print(f'  {var_name:>3s}: {status}')

In [ ]:
# ── 单样本数值范围检查 ────────────────────────────────────────────────────────
x_phys     = vis_cache['x_phys']      # [B, C, H, W]
recon_phys = vis_cache['recon_phys']
mask_bool  = mask.bool()              # [C, H, W]
SAMPLE_IDX = 0

for ch in [0, 25, 50, 75, 100]:
    if ch >= x_phys.shape[1]:
        break
    mk     = mask_bool[ch] if mask_bool.ndim == 3 else mask_bool
    gt_oc  = x_phys[SAMPLE_IDX, ch][mk]
    rc_oc  = recon_phys[SAMPLE_IDX, ch][mk]
    print(
        f'CH{ch:3d}  input [{gt_oc.min():.3f}, {gt_oc.max():.3f}]  '
        f'recon [{rc_oc.min():.3f}, {rc_oc.max():.3f}]  '
        f'MAE={torch.mean(torch.abs(rc_oc - gt_oc)):.4f}'
    )

In [ ]:
# ── 指标分布直方图 ───────────────────────────────────────────────────────────
from matplotlib.ticker import MaxNLocator

fig, axes = plt.subplots(1, 5, figsize=(20, 3))
for ax, col in zip(axes, ['mse', 'mae', 'rmse', 'r2', 'psnr']):
    ax.hist(df[col].dropna().values, bins=20)
    ax.set_title(col)
    ax.yaxis.set_major_locator(MaxNLocator(integer=True))
plt.tight_layout()
plt.show()

# ── 按变量分组的 R² 对比 ────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
var_r2 = df_var.groupby('variable')['r2'].mean()
colors = {'U': 'tab:blue', 'V': 'tab:orange', 'T': 'tab:green', 'S': 'tab:red', 'SSH': 'tab:purple'}
bars = ax.bar(var_r2.index, var_r2.values,
              color=[colors.get(v, 'gray') for v in var_r2.index])
ax.set_ylabel('R²')
ax.set_title('Per-variable R²')
ax.set_ylim(min(0, var_r2.min() - 0.05), 1.05)
ax.axhline(0, color='black', lw=0.5)
for bar, val in zip(bars, var_r2.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.4f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# ── 逐通道 Pearson 相关系数 ──────────────────────────────────────────────────
C = len(corr_mean)
fig, ax = plt.subplots(figsize=(max(14, C * 0.15), 4))

# 按变量着色
ch_colors = []
for ch in range(C):
    if ch < 25:    ch_colors.append('tab:blue')    # U
    elif ch < 50:  ch_colors.append('tab:orange')  # V
    elif ch < 75:  ch_colors.append('tab:green')   # T
    elif ch < 100: ch_colors.append('tab:red')     # S
    else:          ch_colors.append('tab:purple')   # SSH

ax.bar(range(C), corr_mean, color=ch_colors, width=1.0, edgecolor='none')
ax.set_xlabel('Channel index')
ax.set_ylabel('Pearson correlation')
ax.set_title('Per-channel reconstruction correlation (sample mean)')
ax.set_xlim(-0.5, C - 0.5)
ax.set_ylim(-1.1, 1.1)
ax.axhline(0, color='black', lw=0.5)
ax.axhline(0.9, color='gray', ls='--', lw=0.8, alpha=0.5, label='0.9')

# 标注负相关通道
for ch in range(C):
    if corr_mean[ch] < 0:
        ax.annotate(CHANNEL_LABELS.get(ch, f'CH{ch}'),
                    (ch, corr_mean[ch]), fontsize=6, ha='center', va='top', rotation=90)

# 变量分隔线
for boundary in [25, 50, 75, 100]:
    if boundary < C:
        ax.axvline(boundary - 0.5, color='gray', lw=0.5, ls=':')

# 图例
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='s', color='w', markerfacecolor='tab:blue', label='U', markersize=8),
    Line2D([0], [0], marker='s', color='w', markerfacecolor='tab:orange', label='V', markersize=8),
    Line2D([0], [0], marker='s', color='w', markerfacecolor='tab:green', label='T', markersize=8),
    Line2D([0], [0], marker='s', color='w', markerfacecolor='tab:red', label='S', markersize=8),
    Line2D([0], [0], marker='s', color='w', markerfacecolor='tab:purple', label='SSH', markersize=8),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=8)
plt.tight_layout()
plt.show()

# 打印问题通道
bad_chs = [(ch, CHANNEL_LABELS.get(ch, f'CH{ch}'), corr_mean[ch])
           for ch in range(C) if corr_mean[ch] < 0.9]
if bad_chs:
    print(f'\n⚠ {len(bad_chs)} channels with correlation < 0.9:')
    for ch, label, r in bad_chs:
        status = 'FLIPPED' if r < 0 else 'WEAK'
        print(f'  {label:>8s} (ch={ch:3d}): r={r:.4f}  [{status}]')
else:
    print('\n✓ All channels have correlation >= 0.9')

In [ ]:
# ── 重建可视化：原场 / 重建场 / 绝对误差 ─────────────────────────────────────
from matplotlib.colors import TwoSlopeNorm

x_np     = vis_cache['x_phys'].numpy()      # [B, C, H, W]
recon_np = vis_cache['recon_phys'].numpy()
mask_np  = mask.numpy().astype(bool)        # [C, H, W]
SAMPLE_IDX = 0

# 默认通道 + 自动加入翻转最严重的通道
max_ch   = x_np.shape[1] - 1
default_chs = sorted(set([0, max_ch // 4, max_ch // 2, 3 * max_ch // 4, max_ch]))
# 加入相关系数最低的通道（如有翻转）
worst_chs = [ch for ch in np.argsort(corr_mean)[:3] if corr_mean[ch] < 0.9]
CHANNELS = sorted(set(default_chs + worst_chs))

def get_mask2d(ch):
    return mask_np[ch] if mask_np.ndim == 3 else mask_np

def masked_field(arr, ch):
    out = arr[SAMPLE_IDX, ch].copy()
    out[~get_mask2d(ch)] = np.nan
    return out

fig, axes = plt.subplots(len(CHANNELS), 3, figsize=(14, 3.5 * len(CHANNELS)))
if len(CHANNELS) == 1:
    axes = axes[np.newaxis]

for r, ch in enumerate(CHANNELS):
    gt_d  = masked_field(x_np, ch)
    rc_d  = masked_field(recon_np, ch)
    err_d = np.abs(rc_d - gt_d)

    ch_label = CHANNEL_LABELS.get(ch, f'CH{ch}')
    corr_val = corr_mean[ch]
    flip_tag = f'  [r={corr_val:.3f}]'


    shared_min = np.nanmin(np.stack([gt_d, rc_d]))
    shared_max = np.nanmax(np.stack([gt_d, rc_d]))
    err_max = max(float(np.nanpercentile(err_d, 99)) if np.isfinite(err_d).any() else 1.0, 1e-8)
    items = [
        (axes[r, 0], gt_d,  f'{ch_label} Input',  'viridis', None, shared_min, shared_max),
        (axes[r, 1], rc_d,  f'{ch_label} Recon{flip_tag}', 'viridis', None, shared_min, shared_max),
        (axes[r, 2], err_d, f'{ch_label} |Error|', 'viridis', None, 0, err_max),
    ]

    for ax, data, title, cmap, norm_obj, vmi, vma in items:
        kw = {'cmap': cmap}
        if norm_obj is not None:
            kw['norm'] = norm_obj
        else:
            kw['vmin'] = vmi
            kw['vmax'] = vma
        im = ax.imshow(data, **kw)
        ax.set_title(title, fontsize=9)
        ax.set_xticks([]); ax.set_yticks([])
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()